# 🏗️ Fabric Data Pipelines Provisioning Notebook

This notebook bootstraps your Fabric pipelines framework into a single target workspace, providing an early visual overview of the pipeline hierarchy before any provisioning steps.

## 📌 Parameters

In [17]:
pipeline_json_path     = "./builtin/Data Engineering Master Updated.json"   # Path to your pipeline template JSON
notebook_file_paths    = []                           # Local `.ipynb` file paths to upload
pipelines_to_deploy    = []                           # [] = all pipelines
visualize_graph        = True                         # Show hierarchy graph before provisioning
auto_provision_missing = True                         # Create placeholders for missing dependencies
workspace_name         = "Data Engineering [Dev]"     # Fabric workspace name to deploy into
replacements = {
    # Map placeholder tokens in your template to actual values
    # e.g., "CONNECTION_GUID_SQL": "<GUID>",
    #       "SQL_FMD_FRAMEWORK": "MyDatabaseName"
}

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 21, Finished, Available, Finished)

## 📥 Imports & Setup

In [18]:
import os, json, time, logging, requests
import sempy_labs.admin as admin
import notebookutils
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential
from graphviz import Digraph
from collections import deque

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger()

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 22, Finished, Available, Finished)

## 🔧 Utility Functions

In [19]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception(
        lambda e: isinstance(e, requests.exceptions.RequestException)
        and getattr(e.request, 'method', '').upper() in ['GET','PUT']
    ),
    reraise=True
)
def fabric_request(session, method, url, **kwargs):
    logger.info(f"{method} {url}")
    r = session.request(method, url, **kwargs)
    if r.status_code == 202 and 'Operation-Location' in r.headers:
        return poll_lro(session, r.headers['Operation-Location'])
    r.raise_for_status()
    return r

def poll_lro(session, op_url, interval=5, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        r = session.get(op_url)
        r.raise_for_status()
        status = r.json().get('status')
        if status == 'Succeeded': return r
        if status == 'Failed': raise Exception("LRO failed")
        time.sleep(interval)
    raise TimeoutError("LRO timeout")

def upsert_item(session, base_url, name, payload):
    r = fabric_request(session, 'GET', base_url)
    names = [i['name'] for i in r.json().get('value', [])]
    if name in names:
        return fabric_request(session, 'PUT', f"{base_url}/{name}", json=payload)
    return fabric_request(session, 'POST', base_url, json=payload)

def deploy_items(session, base_url, items):
    for it in items:
        upsert_item(session, base_url, it['name'], {'name': it['name'], 'properties': it['properties']})
        logger.info(f"Deployed: {it['name']}")

def create_session(token):
    headers = {'Authorization': f"Bearer {token}", 'Content-Type': 'application/json'}
    sess = requests.Session()
    sess.headers.update(headers)
    return sess

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 23, Finished, Available, Finished)

## 📄 Load & Prepare Pipeline Template

In [20]:
# Load JSON template once
with open(pipeline_json_path) as f:
    tpl = json.load(f)
# Raw serialization for placeholder replacement
tpl_raw = json.dumps(tpl)

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 24, Finished, Available, Finished)

## 🌐 Resolve Workspace & Authenticate

In [21]:
# Resolve workspace ID via sempy_labs
df = admin.list_workspaces(workspace=workspace_name)
if df.empty:
    raise ValueError(f"Workspace not found: {workspace_name}")
workspace_id = df.iloc[0]['Id']
logger.info(f"Workspace ID: {workspace_id}")

# Authenticate using built-in identity
token = notebookutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
if not token:
    raise Exception("Failed to acquire Fabric access token")
session = create_session(token)

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 25, Finished, Available, Finished)

2025-06-18 02:17:16,807 INFO Workspace ID: fe3df7f0-d348-4f65-9c4f-9286a370a09e


## 🔄 Preprocess & Normalize Pipelines

In [22]:
# Build pipeline definitions
data = json.loads(tpl_raw)
pipelines = []
for r in data.get('resources', []):
    if r.get('type') != 'pipelines':
        continue
    # Apply replacements on properties
    props = r['properties']
    props_str = json.dumps(props)
    for ph, val in replacements.items():
        props_str = props_str.replace(ph, val)
    np = json.loads(props_str)
    # Reset ID and enforce workspace
    np.pop('pipelineId', None)
    np['workspaceId'] = workspace_id
    r['properties'] = np
    pipelines.append(r)
logger.info(f"Prepared {len(pipelines)} pipelines")

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 26, Finished, Available, Finished)

2025-06-18 02:17:21,916 INFO Prepared 12 pipelines


## 📊 Visualize Pipeline Hierarchy

In [24]:
if visualize_graph:
    # Build parent→child map
    hierarchy = {p['name']:[a['name'] for a in p['properties'].get('activities',[]) if a['type']=='InvokePipeline'] for p in pipelines}
    # Print indented tree view
    print("Pipeline Hierarchy:")
    for parent, children in hierarchy.items():
        print(parent)
        if children:
            for child in children:
                print(f"  └─ {child}")
        else:
            print("  └─ (no child pipelines)")
    # Display as DataFrame
    import pandas as pd
    rows = []
    for parent, children in hierarchy.items():
        if children:
            for child in children:
                rows.append({"parent": parent, "child": child})
        else:
            rows.append({"parent": parent, "child": None})
    display(pd.DataFrame(rows))

StatementMeta(, 4254edd9-ad23-4f04-8775-80c9cae51699, 28, Finished, Available, Finished)

Pipeline Hierarchy:
Data Engineering Master
  └─ Data Engineering
Data Engineering
  └─ Bronze
  └─ Silver
  └─ Gold
  └─ Semantic
Bronze Master
  └─ Update Control Table
Silver Master
  └─ Update Control Table
Gold Master
  └─ Update Control Table
Semantic Models
  └─ (no child pipelines)
Update Bronze Control Table
  └─ (no child pipelines)
Update Silver Control Table
  └─ (no child pipelines)
Update Gold Control Table
  └─ (no child pipelines)
Database Source - Extract Pipeline
  └─ (no child pipelines)
Database Source - Bronze to Silver
  └─ (no child pipelines)
Database Source - Silver To Gold
  └─ (no child pipelines)


SynapseWidget(Synapse.DataFrame, d5d2f4a3-475d-4ae8-b24d-f3266cb664be)

## ⚙️ Dependency Validation & Provision

In [ ]:
# Check notebooks & connections
nb_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/notebooks"
cn_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/connections"
exist_nbs = {n['name'] for n in fabric_request(session, 'GET', nb_url).json().get('value', [])}
exist_conns = {c['name'] for c in fabric_request(session, 'GET', cn_url).json().get('value', [])}

missing_nbs, missing_conns = set(), set()
for p in pipelines:
    for act in p['properties'].get('activities', []):
        tp = act.get('typeProperties', {})
        if 'notebook' in tp:
            ref = tp['notebook']['referenceName']
            if ref not in exist_nbs:
                missing_nbs.add(ref)
        for key in ('source', 'sink', 'dataset', 'lookup', 'storedProcedure'):
            ref = tp.get(key, {}).get('referenceName')
            if ref and ref not in exist_conns:
                missing_conns.add(ref)

# Auto-provision if enabled
def provision_placeholders(names, url, kind):
    for nm in names:
        if auto_provision_missing:
            payload = {
                'name': nm, 
                'properties': {
                    'content': '# Placeholder' if kind=='notebook' else {}, 
                    'type': 'PlaceholderConnection' if kind=='connection' else None
                }
            }
            fabric_request(session, 'POST', url, json=payload)
            logger.info(f"Provisioned {kind}: {nm}")
        else:
            raise RuntimeError(f"Missing {kind}s: {names}")

if missing_nbs:
    provision_placeholders(missing_nbs, nb_url, 'notebook')
if missing_conns:
    provision_placeholders(missing_conns, cn_url, 'connection')

## 🔎 Select Pipelines to Deploy

In [ ]:
def get_descendants(root, map):
    desc, q = set(), deque([root])
    while q:
        cur = q.popleft()
        for ch in map.get(cur, []):
            if ch not in desc:
                desc.add(ch)
                q.append(ch)
    return desc

if pipelines_to_deploy:
    selected, miss = set(), []
    for t in pipelines_to_deploy:
        if t not in hierarchy:
            miss.append(t)
        else:
            selected.add(t)
            selected |= get_descendants(t, hierarchy)
    if miss:
        raise ValueError(f"Missing pipelines: {miss}")
    pipelines = [p for p in pipelines if p['name'] in selected]
    logger.info(f"Deploying {len(pipelines)} pipelines")
else:
    logger.info("Deploying all pipelines")

## 🚀 Deploy Pipelines

In [ ]:
base_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/pipelines"
deploy_items(session, base_url, pipelines)
logger.info("Deployment complete!")

## 🎉 Completed
All pipelines have been deployed to **{workspace_name}** successfully!